# N-BaIoT Preprocessing — Step 1: Load & Inspect
**Device: Danmini Doorbell**

This notebook reads the extracted CSV files directly. It does not read or extract the `.rar` archives.
It locates `Danmini_Doorbell` relative to the notebook's working directory and searches recursively for the `gafgyt_attacks` and `mirai_attacks` folders.

### What this notebook does:
1. Load benign traffic (`benign_traffic.csv`)
2. Load extracted Gafgyt and Mirai attack CSVs
3. Assign ground-truth labels to each traffic class
4. Combine and inspect the labelled data
5. Save a Step 1 checkpoint


In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# ===== CONFIGURATION =====
DEVICE = "Danmini_Doorbell"

# Find the device folder relative to the notebook's current working directory.
# If it is not found automatically, set BASE_DIR manually to the folder
# that CONTAINS Danmini_Doorbell.
cwd = Path.cwd()
search_roots = [cwd, *cwd.parents]
DEVICE_DIR = None

# First check the current directory and its parent directories.
for root in search_roots:
    candidate = root / DEVICE
    if candidate.is_dir():
        DEVICE_DIR = candidate
        break

# If needed, search below the current working directory.
if DEVICE_DIR is None:
    matches = [p for p in cwd.rglob(DEVICE) if p.is_dir()]
    if matches:
        DEVICE_DIR = matches[0]

if DEVICE_DIR is None:
    raise FileNotFoundError(
        f"Could not find '{DEVICE}'. Current working directory is: {cwd}\n"
        "Move the notebook into the dataset project folder, or manually set "
        "DEVICE_DIR to the full path of the Danmini_Doorbell folder."
    )

BASE_DIR = DEVICE_DIR.parent

print(f"Current working directory : {cwd}")
print(f"Dataset root               : {BASE_DIR}")
print(f"Device folder              : {DEVICE_DIR}")
print("\\nDevice folder contents:")
for item in sorted(DEVICE_DIR.iterdir()):
    print(f"  {'📁' if item.is_dir() else '📄'} {item.name}{'/' if item.is_dir() else ''}")


---
## 1.1 Load Benign Traffic

In [ ]:
benign_path = os.path.join(DEVICE_DIR, 'benign_traffic.csv')
print(f'Loading benign traffic from: {benign_path} ...')

df_benign = pd.read_csv(benign_path)
df_benign['label'] = 'benign'

print(f'✓ Loaded benign traffic!')
print(f'  Rows     : {df_benign.shape[0]:,}')
print(f'  Features : {df_benign.shape[1] - 1}')
print(f'  Memory   : {df_benign.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
df_benign.head(3)

In [ ]:
# Verify column count and datatypes for benign traffic
feature_cols = [c for c in df_benign.columns if c != 'label']
print(f'Feature columns count: {len(feature_cols)} (Expected: 115)')
print('Data types breakdown:')
print(df_benign[feature_cols].dtypes.value_counts())

---
## 1.2 Load Extracted Attack CSVs Directly

Since the attack CSV files are already uncompressed in:
- `Danmini_Doorbell/gafgyt_attacks/` (`combo.csv`, `junk.csv`, `scan.csv`, `tcp.csv`, `udp.csv`)
- `Danmini_Doorbell/mirai_attacks/` (`ack.csv`, `scan.csv`, `syn.csv`, `udp.csv`, `udpplain.csv`)

We read each CSV file directly, append a distinct ground-truth `label` column (e.g. `gafgyt_combo`, `mirai_ack`), and aggregate them.

In [ ]:
attack_dfs = []

# Find the extracted attack folders by their actual folder names,
# wherever they are nested inside Danmini_Doorbell.
botnet_configs = [
    {"name": "gafgyt", "folder": "gafgyt_attacks"},
    {"name": "mirai",  "folder": "mirai_attacks"},
]

for cfg in botnet_configs:
    botnet = cfg["name"]
    folder_name = cfg["folder"]

    matches = [
        p for p in DEVICE_DIR.rglob(folder_name)
        if p.is_dir()
    ]

    if not matches:
        print(f"⚠️ Could not find extracted folder '{folder_name}' under {DEVICE_DIR}")
        continue

    # Use the first matching extracted folder.
    folder = matches[0]
    csv_files = sorted(folder.glob("*.csv"))

    print(f"\\n📂 Loading {botnet.upper()} attacks from: {folder}")
    print(f"   CSV files found: {len(csv_files)}")

    if not csv_files:
        print(f"⚠️ No CSV files found directly inside {folder}")
        continue

    for csv_file in csv_files:
        attack_name = csv_file.stem
        label = f"{botnet}_{attack_name}"

        df_atk = pd.read_csv(csv_file)
        df_atk["label"] = label
        attack_dfs.append(df_atk)

        size_mb = csv_file.stat().st_size / (1024 ** 2)
        print(
            f"  ✓ {label:22s} : "
            f"{len(df_atk):>10,} rows ({size_mb:5.1f} MB)"
        )

print(f"\\nTotal attack CSV files loaded: {len(attack_dfs)}")

if not attack_dfs:
    raise RuntimeError(
        "No attack CSV files were loaded. Check that gafgyt_attacks and "
        "mirai_attacks have been extracted under Danmini_Doorbell."
    )


---
## 1.3 Combine All Traffic Data

In [ ]:
print('Combining benign and attack datasets...')
df_device = pd.concat([df_benign] + attack_dfs, ignore_index=True)

print(f'\n=== Combined Dataset for {DEVICE} ===')
print(f'Shape         : {df_device.shape}')
print(f'Total Samples : {df_device.shape[0]:,}')
print(f'Features      : {df_device.shape[1] - 1}')
print(f'Memory Usage  : {df_device.memory_usage(deep=True).sum() / (1024**2):.1f} MB')

print(f'\nClass Distribution:')
dist = df_device['label'].value_counts()
for lbl, cnt in dist.items():
    pct = (cnt / len(df_device)) * 100
    print(f'  {lbl:22s} : {cnt:>8,}  ({pct:5.2f}%)')

---
## 1.4 Quality Inspection & Sanity Checks

In [ ]:
feature_cols = [c for c in df_device.columns if c != 'label']

# 1. Missing Values (NaN/Null)
null_counts = df_device[feature_cols].isnull().sum()
total_nulls = null_counts.sum()
print(f'1. Total Missing Values (NaN) : {total_nulls:,}')
if total_nulls > 0:
    print('   Columns with NaNs:')
    print(null_counts[null_counts > 0])
else:
    print('   ✓ No missing values found.')

# 2. Infinite Values
inf_mask = np.isinf(df_device[feature_cols])
total_infs = inf_mask.sum().sum()
print(f'\n2. Total Infinite Values (±Inf): {total_infs:,}')
if total_infs > 0:
    inf_by_col = inf_mask.sum()
    print('   Columns with Infs:')
    print(inf_by_col[inf_by_col > 0])
else:
    print('   ✓ No infinite values found.')

# 3. Verify numeric data type
non_num = df_device[feature_cols].select_dtypes(exclude=[np.number]).columns.tolist()
print(f'\n3. Non-numeric features       : {len(non_num)}')
if len(non_num) == 0:
    print('   ✓ All 115 features are numeric.')
else:
    print(f'   ⚠️ Non-numeric columns: {non_num}')

# 4. Duplicate Rows (identical feature vectors)
dups = df_device.duplicated(subset=feature_cols).sum()
dup_pct = (dups / len(df_device)) * 100
print(f'\n4. Duplicate rows (features)  : {dups:,} ({dup_pct:.2f}%)')
print('   (Note: Network stream statistics frequently repeat idle packet sizes/states; handled in cleaning)')

In [ ]:
# Verify Feature Group Breakdown per N-BaIoT Paper specification
# MI_dir (15), H (15), HH (35), HH_jit (15), HpHp (35) = 115 total
groups = {}
for col in feature_cols:
    prefix = col.split('_L')[0]
    groups[prefix] = groups.get(prefix, 0) + 1

print('Feature Groups check:')
for g_name, cnt in sorted(groups.items()):
    print(f'  {g_name:12s} : {cnt:3d} features')
print(f'  {"TOTAL":12s} : {sum(groups.values()):3d}')

---
## 1.5 Checkpoint — Save Raw Labelled Dataset

We save the combined dataset with the `label` column to disk as a `.csv` or `.parquet`.
This checkpoint will be loaded directly into **Step 2: Clean** without having to reload the 11 separate attack files.

In [ ]:
OUTPUT_DIR = os.path.join(BASE_DIR, 'preprocessed')
os.makedirs(OUTPUT_DIR, exist_ok=True)

checkpoint_path = os.path.join(OUTPUT_DIR, f'{DEVICE}_step1_raw_labelled.csv')
print(f'Saving checkpoint to: {checkpoint_path} ...')
df_device.to_csv(checkpoint_path, index=False)

size_mb = os.path.getsize(checkpoint_path) / (1024**2)
print(f'✓ Checkpoint successfully saved!')
print(f'  Path     : {checkpoint_path}')
print(f'  Size     : {size_mb:.1f} MB')
print(f'  Rows     : {len(df_device):,}')
print(f'  Classes  : {df_device["label"].nunique()} classes')
print('\n🚀 Step 1 Complete! Ready for Step 2: Data Cleaning.')